In [1]:
from automated_events_coding.pipeline import (
    step0_relevance,
    step1_events,
    step2_location,
    step3_event_lead,
    step4_interaction,
    step5_identification,
)
from automated_events_coding.pipeline.schemas import Event, EventList, EventLead, EventRecord, InteractionType
from automated_events_coding.llm.client import get_client
from automated_events_coding.llm.server import LlamaServerConfig, LlamaServer
from example_articles import ARTICLES, ARTICLES_BY_CATEGORY
from automated_events_coding.pipeline.prompts import STEP1_EVENT_EXTRACTION, STEP2_LOCATION_EXTRACTION, STEP3_EVENT_LEAD

import re
import json
from pydantic import TypeAdapter
from pprint import pprint


In [4]:
article_text = ARTICLES["018_E1"]
article_text

'UNITED NATIONS, May 12 -- The United Nations Educational, Scientific and Cultural Organization issued the following news release: The UNESCO Regional Office for Southern Africa will hold an inception workshop for the newly launched project, "A community focused flood Early Warning System for the Buzi-Pungwe and Save (BuPuSa) Transboundary River Basins" in Maputo, Mozambique on 18 May 2022. The 3-year project is being implemented in Mozambique and Zimbabwe by UNESCO and funded by the Austrian Development Agency (ADA) with funds of the Austrian Development Cooperation. The project was conceptualised in response to the observed increased frequency, duration and intensity of natural disasters and extreme weather conditions in recent years, such as landslides, floods, tropical cyclones, droughts and prolonged dry spells that negatively impact the livelihoods of the people in the two countries. The project therefore seeks to reduce the vulnerability of communities in the BuPuSa transboundar

In [ ]:
relevance = step0_relevance.check_relevance(article_text=article_text)
relevance

In [5]:
from automated_events_coding.pipeline.prompts import PROJECT_BOILERPLATE, STEP1_EVENT_EXTRACTION, STEP2_LOCATION_EXTRACTION, STEP3_EVENT_LEAD
import re
import json
from pydantic import TypeAdapter
from pprint import pprint

# json_pattern = r"\[.*?\]"
json_pattern = r"```json(.*?)```"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

with LlamaServer(llama_server_config) as server:
    client = get_client(llama_server_config)

    # step 1
    response = client.chat.completions.create(
        model="Test",
        messages=[
            {"role": "system", "content": STEP1_EVENT_EXTRACTION.replace("[pb]", PROJECT_BOILERPLATE)},
            {"role": "user", "content": article_text},
        ],
    )
    raw = response.choices[0].message.content
    content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
    events = []
    for event in content:
        event_obj = Event(text=event["event_summary"])
        events.append(event_obj)
    validated = EventList(events=events)
    for i, event in enumerate(validated.events):
        print(f"Event {i+1}")
        pprint(event.text)
        print("="*20)

    # # step 2
    # basin_event_branches = {}
    # for i, event in enumerate(validated.events):
    #     response = client.chat.completions.create(
    #         model="Test",
    #         messages=[
    #             {"role": "system", "content": STEP2_LOCATION_EXTRACTION},
    #             {"role": "user", "content": event.text},
    #         ],
    #     )
    #     raw = response.choices[0].message.content
    #     content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
    #     print("="*20)
    #     print("Placenames found:")
    #     print(content)
    #     validated_basins = step2_location.lookup_country_basin(content)
    #     print("Basins found:")
    #     print(validated_basins)
    #     print("="*20)
    #     for j, basin in enumerate(validated_basins):
    #         event_id = f"Event{i+1}_{basin.bcode}_{j+1}"
    #         basin_event_branches[event_id] = (event, basin)


    # # step 3
    # print("="*20)
    # event_template = "# Event: {event}\n# Article: {article}"
    # event_led_extraction = []
    # for i, event in enumerate(validated.events):
    #     response = client.chat.completions.create(
    #         model="Test",
    #         messages=[
    #             {"role": "system", "content": STEP3_EVENT_LEAD},
    #             {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
    #         ],
    #     )
    #     raw = response.choices[0].message.content
    #     content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
    #     print(event.text)
    #     print(content)
    #     print("="*20)
    #     event_led_extraction += content

    # # step 4
    # print("="*20)
    # pprint(basin_event_branches)
    # print("="*20)

Event 1
('UNESCO issued a news release announcing that the UNESCO Regional Office for '
 'Southern Africa will hold an inception workshop for the newly launched '
 "project 'A community focused flood Early Warning System for the Buzi-Pungwe "
 "and Save (BuPuSa) Transboundary River Basins' in Maputo, Mozambique on 18 "
 'May 2022. The 3-year project, implemented in Mozambique and Zimbabwe by '
 'UNESCO and funded by the Austrian Development Agency (ADA) with funds of the '
 'Austrian Development Cooperation, seeks to reduce community vulnerability to '
 'floods by providing flood risk assessment, monitoring, early warning '
 'systems, and capacity building.')


In [6]:
content

[{'event_summary': "UNESCO issued a news release announcing that the UNESCO Regional Office for Southern Africa will hold an inception workshop for the newly launched project 'A community focused flood Early Warning System for the Buzi-Pungwe and Save (BuPuSa) Transboundary River Basins' in Maputo, Mozambique on 18 May 2022. The 3-year project, implemented in Mozambique and Zimbabwe by UNESCO and funded by the Austrian Development Agency (ADA) with funds of the Austrian Development Cooperation, seeks to reduce community vulnerability to floods by providing flood risk assessment, monitoring, early warning systems, and capacity building.",
  'placenames': [{'placename': 'United Nations',
    'placename_type': 'non-country'},
   {'placename': 'Mozambique', 'placename_type': 'country'},
   {'placename': 'Buzi-Pungwe', 'placename_type': 'non-country'},
   {'placename': 'Save', 'placename_type': 'non-country'},
   {'placename': 'BuPuSa', 'placename_type': 'non-country'},
   {'placename': 'Ma

In [ ]:
from automated_events_coding.pipeline.prompts import STEP4_1_ENTITY_EXTRACTION, STEP4_3_DYAD_EXTRACTION 
import pandas as pd
from pathlib import Path
import itertools

_GAZETTEER_PATH = Path("automated_events_coding") / "static" / "Dictionaries_20260803.xlsx"
_BASIN_SHEET = "Basin Names LU"

def look_riparian_countries(bcode):
    basins = pd.read_excel(_GAZETTEER_PATH, sheet_name=_BASIN_SHEET, engine="openpyxl")
    return basins[basins.BCODE == bcode]["Riparian Countries"].values[0].split(";")

# json_pattern = r"\[.*?\]"
json_pattern = r"```json(.*?)```"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

with LlamaServer(llama_server_config) as server:
    for key, value in basin_event_branches.items():
        event, lu_basin = value
        print("="*20)
        basin = key.split("_")[1]
        print(basin)
        riparian_countries = look_riparian_countries(basin)
        pprint(riparian_countries)
        entity_extraction_template = "# Event:\n {event}\n\n# Riparian Counties:\n{rip_countries}\n\n# Article Text:\n {article}\n\n"
        response = client.chat.completions.create(
            model="Test",
            messages=[
                {"role": "system", "content": STEP4_1_ENTITY_EXTRACTION},
                {"role": "user", "content": entity_extraction_template.format(event=event.text, rip_countries=riparian_countries, article=article_text)},
            ],
        )
        raw = response.choices[0].message.content
        content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        pprint(event.text)
        pprint(content)

        # # create rank-2 perms
        # dyad_template = "# Event:\n {event}\n\n# Entities: {perm}"
        # rank2_perms = list(itertools.permutations(content, r=2))
        # for perm in rank2_perms:
        #     print(perm)
        #     response = client.chat.completions.create(
        #         model="Test",
        #         messages=[
        #             {"role": "system", "content": STEP4_3_DYAD_EXTRACTION},
        #             {"role": "user", "content": dyad_template.format(event=event.text, perm=perm)},
        #         ],
        #     )
        #     raw = response.choices[0].message.content
        #     content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        #     pprint(content)
        #     print("="*20)

# As function

In [ ]:
json_pattern = r"\[.*?\]"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

def full_process(article_text, llama_server_config=llama_server_config):
    "Full process so far..."
    relevance = step0_relevance.check_relevance(article_text=article_text)
    if relevance.meta['relevance_label'] != 'Y':
        print("!! ARTICLE IRRELEVANT !!")

    event_template = "# Event: {event}\n# Article: {article}"
    with LlamaServer(llama_server_config) as server:
        client = get_client(llama_server_config)

        # step 1
        response = client.chat.completions.create(
            model="Test",
            messages=[
                {"role": "system", "content": STEP1_EVENT_EXTRACTION},
                {"role": "user", "content": article_text},
            ],
        )
        raw = response.choices[0].message.content
        content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        events = []
        for event in content:
            event_obj = Event(text=event["event_summary"])
            events.append(event_obj)
        validated = EventList(events=events)
        for i, event in enumerate(validated.events):
            print(f"Event {i+1}")
            pprint(event.text)
            print("="*20)

        # step 2
        placenames = []
        for i, event in enumerate(validated.events):
            response = client.chat.completions.create(
                model="Test",
                messages=[
                    {"role": "system", "content": STEP2_LOCATION_EXTRACTION},
                    {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
                ],
            )
            raw = response.choices[0].message.content
            content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
            placenames += content
        print("="*20)
        print("Placenames found:")
        print(placenames)
        validated_basins = step2_location.lookup_country_basin(placenames)
        print("Basins found:")
        print(validated_basins)
        print("="*20)

        # step 3
        print("="*20)
        event_led_extraction = []
        for i, event in enumerate(validated.events):
            response = client.chat.completions.create(
                model="Test",
                messages=[
                    {"role": "system", "content": STEP3_EVENT_LEAD},
                    {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
                ],
            )
            raw = response.choices[0].message.content
            content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
            pprint(event.text)
            print(content)
            print("="*20)
            event_led_extraction += content

    return validated, validated_basins, event_led_extraction

In [ ]:
article_text = ARTICLES["028_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)

In [ ]:
article_text = ARTICLES["018_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)

In [ ]:
article_text = ARTICLES["065_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)